# Single-Task GNN Models

## Scientific objective
Train a compact GIN (or configured GAT) per endpoint on 2D molecular graphs using early stopping and identical scaffold partitions.

## Inputs
- Standardized SMILES and scaffold splits
- Graph config

## Expected outputs
- `models/gnn/*_gnn.pt`
- `results/metrics/single_task_gnn.csv`
- explicit dependency/status report

## Dependencies
PyTorch Geometric, PyTorch

## Reproducibility seed
`20260723`. The seed is loaded from `configs/training_config.yaml`; split files and checkpoints are persisted.

## Data and model assumptions
The first graph architecture is intentionally modest. GNNs do not automatically solve scaffold shift or activity cliffs.

## Validation checks
The executable cells below fail explicitly on missing/inconsistent required artifacts and save machine-readable status records.

## Interpretation of results
Interpret endpoint-level outputs only after checking prevalence, missingness, split integrity, calibration, uncertainty, and applicability-domain coverage. No notebook result is evidence that experimental toxicity testing can be replaced.

## Saved artifacts
Artifacts listed above are written under `data/`, `models/`, `results/`, `figures/`, `tables/`, or `reports/` and are consumed by later notebooks.

## Limitations
PyTorch Geometric installation is platform-sensitive. A missing dependency produces a documented failure, not a silent skip.

## Next notebook
[13_multitask_fingerprint_model.ipynb](./13_multitask_fingerprint_model.ipynb)


In [1]:
from pathlib import Path
import os, json, warnings
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "pyproject.toml").exists():
    raise RuntimeError("Run this notebook from the repository root or notebooks directory")
os.chdir(ROOT)

from toxicity_screening.config import load_configs, execution_profile
from toxicity_screening.utils import set_global_seed, require_paths

CONFIGS = load_configs(ROOT)
PROFILE, PROFILE_CONFIG = execution_profile(CONFIGS)
SEED = int(CONFIGS["training_config"]["seed"])
set_global_seed(SEED)
print({"root": str(ROOT), "profile": PROFILE, "seed": SEED})


{'root': 'D:\\Dropbox\\Work\\Learning\\Python\\toxicity_screening_project', 'profile': 'full', 'seed': 20260723}


In [ ]:
# Single-task GNN training with endpoint progress, epoch progress,
# elapsed time, and estimated remaining time.

from toxicity_screening.utils import atomic_write_json

import copy
import time
from datetime import datetime, timedelta

import numpy as np
import pandas as pd


# ------------------------------------------------------------------
# Dependencies
# ------------------------------------------------------------------

try:
    import torch
    from torch import nn

    from torch_geometric.loader import DataLoader as GraphDataLoader

    from toxicity_screening.datasets import MolecularGraphDataset
    from toxicity_screening.gnn_models import GraphClassifier
    from toxicity_screening.graph_features import ATOM_FEATURE_DIM
    from toxicity_screening.metrics import binary_metrics
    from toxicity_screening.training import (
        TrainingResult,
        resolve_device,
    )

except ImportError as exc:
    atomic_write_json(
        {
            "status": "dependency_unavailable",
            "error": str(exc),
            "alternative": (
                "Install environment.yml on a supported "
                "PyTorch/PyG platform and rerun."
            ),
        },
        ROOT / "reports/gnn_dependency_status.json",
    )
    raise


# Optional stability controls for this Windows environment.
torch.set_num_threads(1)

try:
    torch.set_num_interop_threads(1)
except RuntimeError:
    pass


# ------------------------------------------------------------------
# Formatting helpers
# ------------------------------------------------------------------

def format_duration(seconds: float) -> str:
    seconds = max(0, int(round(seconds)))
    return str(timedelta(seconds=seconds))


def current_clock() -> str:
    return datetime.now().strftime("%H:%M:%S")


# ------------------------------------------------------------------
# Training implementation with live epoch reporting
# ------------------------------------------------------------------

def train_graph_binary_model_with_progress(
    model,
    train_loader,
    validation_loader,
    *,
    endpoint: str,
    endpoint_number: int,
    endpoint_total: int,
    epochs: int,
    patience: int,
    learning_rate: float,
    weight_decay: float,
    gradient_clip_norm: float,
    positive_weight: float | None = None,
    checkpoint_path=None,
    device: str = "auto",
):
    device_object = resolve_device(device)
    model.to(device_object)

    criterion = nn.BCEWithLogitsLoss(
        pos_weight=(
            torch.tensor(
                float(positive_weight),
                device=device_object,
            )
            if positive_weight is not None
            else None
        )
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=learning_rate,
        weight_decay=weight_decay,
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.5,
        patience=max(1, patience // 3),
    )

    best_loss = float("inf")
    best_epoch = 0
    best_state = None
    wait = 0
    history = []

    endpoint_start = time.perf_counter()
    epoch_durations = []

    def run_epoch(loader, training: bool) -> float:
        model.train(training)
        losses = []

        for batch in loader:
            batch = batch.to(device_object)
            target = batch.y.view(-1).float()

            if training:
                optimizer.zero_grad(
                    set_to_none=True
                )

            logits = model(batch)
            loss = criterion(
                logits,
                target,
            )

            if training:
                loss.backward()

                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    gradient_clip_norm,
                )

                optimizer.step()

            losses.append(
                float(
                    loss.detach().cpu()
                )
            )

        if not losses:
            raise RuntimeError(
                f"{endpoint}: loader produced no batches"
            )

        return float(np.mean(losses))

    print(
        f"\n[GNN] endpoint_started "
        f"endpoint={endpoint} "
        f"endpoint_progress={endpoint_number}/{endpoint_total} "
        f"device={device_object} "
        f"maximum_epochs={epochs} "
        f"patience={patience} "
        f"time={current_clock()}",
        flush=True,
    )

    for epoch in range(1, epochs + 1):
        epoch_start = time.perf_counter()

        train_loss = run_epoch(
            train_loader,
            training=True,
        )

        with torch.no_grad():
            validation_loss = run_epoch(
                validation_loader,
                training=False,
            )

        scheduler.step(
            validation_loss
        )

        epoch_duration = (
            time.perf_counter()
            - epoch_start
        )

        epoch_durations.append(
            epoch_duration
        )

        history.append(
            {
                "epoch": float(epoch),
                "train_loss": train_loss,
                "validation_loss": validation_loss,
                "learning_rate": float(
                    optimizer.param_groups[0]["lr"]
                ),
                "epoch_seconds": float(
                    epoch_duration
                ),
            }
        )

        improved = (
            validation_loss
            < best_loss - 1e-6
        )

        if improved:
            best_loss = validation_loss
            best_epoch = epoch

            best_state = copy.deepcopy(
                model.state_dict()
            )

            wait = 0
        else:
            wait += 1

        mean_epoch_time = float(
            np.mean(epoch_durations)
        )

        nominal_epochs_remaining = (
            epochs - epoch
        )

        endpoint_eta_seconds = (
            mean_epoch_time
            * nominal_epochs_remaining
        )

        elapsed_endpoint = (
            time.perf_counter()
            - endpoint_start
        )

        print(
            f"[GNN] epoch_completed "
            f"endpoint={endpoint} "
            f"epoch={epoch}/{epochs} "
            f"train_loss={train_loss:.6f} "
            f"validation_loss={validation_loss:.6f} "
            f"best_validation_loss={best_loss:.6f} "
            f"best_epoch={best_epoch} "
            f"wait={wait}/{patience} "
            f"epoch_time={format_duration(epoch_duration)} "
            f"endpoint_elapsed={format_duration(elapsed_endpoint)} "
            f"nominal_endpoint_eta={format_duration(endpoint_eta_seconds)}",
            flush=True,
        )

        if wait >= patience:
            print(
                f"[GNN] early_stopping "
                f"endpoint={endpoint} "
                f"epoch={epoch} "
                f"best_epoch={best_epoch} "
                f"best_validation_loss={best_loss:.6f}",
                flush=True,
            )
            break

    if best_state is None:
        raise RuntimeError(
            f"{endpoint}: training did not produce a checkpoint"
        )

    model.load_state_dict(
        best_state
    )

    checkpoint = None

    if checkpoint_path is not None:
        checkpoint_path.parent.mkdir(
            parents=True,
            exist_ok=True,
        )

        torch.save(
            {
                "state_dict": best_state,
                "best_epoch": best_epoch,
                "history": history,
            },
            checkpoint_path,
        )

        checkpoint = str(
            checkpoint_path
        )

    endpoint_elapsed = (
        time.perf_counter()
        - endpoint_start
    )

    print(
        f"[GNN] training_completed "
        f"endpoint={endpoint} "
        f"epochs_completed={len(history)} "
        f"best_epoch={best_epoch} "
        f"best_validation_loss={best_loss:.6f} "
        f"elapsed={format_duration(endpoint_elapsed)} "
        f"checkpoint={checkpoint}",
        flush=True,
    )

    return TrainingResult(
        best_epoch=best_epoch,
        best_validation_loss=best_loss,
        history=history,
        checkpoint_path=checkpoint,
    )


# ------------------------------------------------------------------
# Load records
# ------------------------------------------------------------------

records = pd.read_parquet(
    ROOT / "data/processed/modeling_records.parquet"
)

endpoint_groups = list(
    records.loc[
        records["label"].notna()
    ].groupby(
        "endpoint",
        sort=False,
    )
)

total_endpoints = len(
    endpoint_groups
)

if total_endpoints == 0:
    raise RuntimeError(
        "No labeled toxicity endpoints were found"
    )

print(
    f"[GNN] full_run_started "
    f"endpoints={total_endpoints} "
    f"profile={PROFILE} "
    f"time={current_clock()}",
    flush=True,
)


# ------------------------------------------------------------------
# Endpoint loop
# ------------------------------------------------------------------

overall_start = time.perf_counter()
completed_endpoint_times = []
rows = []

for endpoint_number, (
    endpoint,
    frame,
) in enumerate(
    endpoint_groups,
    start=1,
):
    endpoint_start = time.perf_counter()

    parts = {
        partition: frame.loc[
            frame["scaffold_split"]
            == partition
        ].copy()
        for partition in [
            "train",
            "validation",
            "test",
        ]
    }

    for partition_name, partition in parts.items():
        if partition.empty:
            raise ValueError(
                f"{endpoint}: {partition_name} partition is empty"
            )

    cap = PROFILE_CONFIG[
        "sample_cap_per_endpoint"
    ]

    train_size = min(
        len(parts["train"]),
        cap or len(parts["train"]),
    )

    train = parts["train"].sample(
        n=train_size,
        random_state=SEED,
    )

    print(
        f"\n[GNN] preparing_endpoint "
        f"endpoint={endpoint} "
        f"progress={endpoint_number}/{total_endpoints} "
        f"train_rows={len(train)} "
        f"validation_rows={len(parts['validation'])} "
        f"test_rows={len(parts['test'])}",
        flush=True,
    )

    loader_start = time.perf_counter()

    loaders = {
        partition_name: GraphDataLoader(
            MolecularGraphDataset(
                partition_frame[
                    "standardized_smiles"
                ],
                partition_frame[
                    "label"
                ].to_numpy(float),
            ),
            batch_size=min(
                64,
                CONFIGS[
                    "training_config"
                ][
                    "batch_size"
                ],
            ),
            shuffle=(
                partition_name
                == "train"
            ),
        )
        for partition_name, partition_frame
        in {
            "train": train,
            "validation": parts[
                "validation"
            ],
        }.items()
    }

    print(
        f"[GNN] loaders_ready "
        f"endpoint={endpoint} "
        f"elapsed={format_duration(time.perf_counter() - loader_start)}",
        flush=True,
    )

    configuration = CONFIGS[
        "model_config"
    ][
        "neural"
    ][
        "gnn"
    ]

    model = GraphClassifier(
        ATOM_FEATURE_DIM,
        configuration["hidden_dim"],
        configuration["layers"],
        configuration["dropout"],
        configuration["architecture"],
    )

    y_train = train[
        "label"
    ].to_numpy(int)

    positive_weight = float(
        (y_train == 0).sum()
        / max(
            1,
            (y_train == 1).sum(),
        )
    )

    result = (
        train_graph_binary_model_with_progress(
            model,
            loaders["train"],
            loaders["validation"],
            endpoint=endpoint,
            endpoint_number=endpoint_number,
            endpoint_total=total_endpoints,
            epochs=PROFILE_CONFIG[
                "max_epochs"
            ],
            patience=PROFILE_CONFIG[
                "patience"
            ],
            learning_rate=CONFIGS[
                "training_config"
            ][
                "optimizer"
            ][
                "learning_rate"
            ],
            weight_decay=CONFIGS[
                "training_config"
            ][
                "optimizer"
            ][
                "weight_decay"
            ],
            gradient_clip_norm=CONFIGS[
                "training_config"
            ][
                "gradient_clip_norm"
            ],
            positive_weight=positive_weight,
            checkpoint_path=(
                ROOT
                / "models"
                / "gnn"
                / f"{endpoint}_gnn.pt"
            ),
        )
    )

    history_path = (
        ROOT
        / "results"
        / "metrics"
        / f"{endpoint}_gnn_history.csv"
    )

    history_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    pd.DataFrame(
        result.history
    ).to_csv(
        history_path,
        index=False,
    )

    print(
        f"[GNN] test_evaluation_started "
        f"endpoint={endpoint}",
        flush=True,
    )

    test_loader = GraphDataLoader(
        MolecularGraphDataset(
            parts["test"][
                "standardized_smiles"
            ],
            parts["test"][
                "label"
            ].to_numpy(float),
        ),
        batch_size=64,
        shuffle=False,
    )

    model.eval()

    device = resolve_device()
    model.to(device)

    probabilities = []

    with torch.no_grad():
        for batch_number, batch in enumerate(
            test_loader,
            start=1,
        ):
            batch = batch.to(device)

            batch_probabilities = (
                torch.sigmoid(
                    model(batch)
                )
                .detach()
                .cpu()
                .numpy()
            )

            probabilities.extend(
                batch_probabilities
            )

            if (
                batch_number == 1
                or batch_number
                % 10 == 0
                or batch_number
                == len(test_loader)
            ):
                print(
                    f"[GNN] test_progress "
                    f"endpoint={endpoint} "
                    f"batch={batch_number}/{len(test_loader)}",
                    flush=True,
                )

    endpoint_metrics = binary_metrics(
        parts["test"][
            "label"
        ].astype(int),
        np.asarray(
            probabilities
        ),
    )

    rows.append(
        {
            "endpoint": endpoint,
            "model": configuration[
                "architecture"
            ],
            "best_epoch": result.best_epoch,
            "best_validation_loss": (
                result.best_validation_loss
            ),
            **{
                key: value
                for key, value
                in endpoint_metrics.items()
                if key
                != "confusion_matrix"
            },
        }
    )

    endpoint_elapsed = (
        time.perf_counter()
        - endpoint_start
    )

    completed_endpoint_times.append(
        endpoint_elapsed
    )

    average_endpoint_time = float(
        np.mean(
            completed_endpoint_times
        )
    )

    endpoints_remaining = (
        total_endpoints
        - endpoint_number
    )

    estimated_remaining = (
        average_endpoint_time
        * endpoints_remaining
    )

    overall_elapsed = (
        time.perf_counter()
        - overall_start
    )

    print(
        f"[GNN] endpoint_completed "
        f"endpoint={endpoint} "
        f"progress={endpoint_number}/{total_endpoints} "
        f"endpoint_elapsed={format_duration(endpoint_elapsed)} "
        f"overall_elapsed={format_duration(overall_elapsed)} "
        f"estimated_remaining={format_duration(estimated_remaining)} "
        f"estimated_finish="
        f"{(datetime.now() + timedelta(seconds=estimated_remaining)).strftime('%H:%M:%S')}",
        flush=True,
    )


# ------------------------------------------------------------------
# Save final metrics
# ------------------------------------------------------------------

gnn_metrics = pd.DataFrame(
    rows
)

metrics_path = (
    ROOT
    / "results"
    / "metrics"
    / "single_task_gnn.csv"
)

gnn_metrics.to_csv(
    metrics_path,
    index=False,
)

total_elapsed = (
    time.perf_counter()
    - overall_start
)

print(
    f"\n[GNN] full_run_completed "
    f"endpoints={len(gnn_metrics)} "
    f"total_elapsed={format_duration(total_elapsed)} "
    f"output={metrics_path} "
    f"time={current_clock()}",
    flush=True,
)

display(
    gnn_metrics
)

[GNN] full_run_started endpoints=6 profile=full time=10:37:46

[GNN] preparing_endpoint endpoint=herg_blockade progress=1/6 train_rows=8971 validation_rows=2095 test_rows=1883
[GNN] loaders_ready endpoint=herg_blockade elapsed=0:00:00

[GNN] endpoint_started endpoint=herg_blockade endpoint_progress=1/6 device=cpu maximum_epochs=150 patience=20 time=10:37:46
[GNN] epoch_completed endpoint=herg_blockade epoch=1/150 train_loss=0.692728 validation_loss=0.662046 best_validation_loss=0.662046 best_epoch=1 wait=0/20 epoch_time=0:00:48 endpoint_elapsed=0:00:48 nominal_endpoint_eta=1:59:29
[GNN] epoch_completed endpoint=herg_blockade epoch=2/150 train_loss=0.649012 validation_loss=0.635652 best_validation_loss=0.635652 best_epoch=2 wait=0/20 epoch_time=0:00:41 endpoint_elapsed=0:01:30 nominal_endpoint_eta=1:50:26
[GNN] epoch_completed endpoint=herg_blockade epoch=3/150 train_loss=0.633445 validation_loss=0.690332 best_validation_loss=0.635652 best_epoch=2 wait=1/20 epoch_time=0:00:44 endpoint_e

### Completion gate
Confirm that the declared artifacts exist before continuing to `13_multitask_fingerprint_model.ipynb`.